# Customer Churn Analysis — Data Preparation

**Purpose:** inspect, clean, validate, and export the customer subscription and monthly usage datasets before analysis in MySQL and Power BI.


## 1. Import libraries and set paths

The notebook expects the raw CSVs in `data/raw`. It creates `data/processed` for the cleaned exports.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"F:\Customer Churn Analysis\05_Data\raw\customer-churn-analysis")

RAW_DIR = PROJECT_DIR
PROCESSED_DIR = Path(r"F:\Customer Churn Analysis\05_Data\Processed")

SUBSCRIPTIONS_FILE = RAW_DIR / "subscriptions.csv"
USAGE_FILE = RAW_DIR / "usage.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Subscriptions exists:", SUBSCRIPTIONS_FILE.exists())
print("Usage exists:", USAGE_FILE.exists())

Subscriptions exists: True
Usage exists: True


## 2. Load the raw datasets

The subscription table contains one record per customer. The usage table contains monthly activity records for customers.

In [2]:
subscriptions = pd.read_csv(SUBSCRIPTIONS_FILE)
usage = pd.read_csv(USAGE_FILE)

print("Subscriptions shape:", subscriptions.shape)
print("Usage shape:", usage.shape)

display(subscriptions.head())
display(usage.head())

Subscriptions shape: (4200, 11)
Usage shape: (30685, 6)


,customer_id,signup_date,plan_type,plan_tier,monthly_price,acquisition_channel,primary_device,age_group,churned,churn_date,cancel_reason
0,1,2024-09-11,monthly,basic,12.99,promo_offer,android,45-54,yes,2025-06-29,technical_issues
1,2,2025-08-14,annual,basic,9.74,promo_offer,web,45-54,no,NaN,NaN
2,3,2025-01-24,monthly,basic,12.99,referral,ios,55+,no,NaN,NaN
3,4,2024-08-15,monthly,basic,12.99,paid_social,ios,25-34,yes,2024-08-20,not_using_enough
4,5,2025-06-24,annual,plus,14.99,paid_social,android,55+,no,NaN,NaN


,customer_id,month,workouts_completed,minutes_active,classes_booked,support_tickets
0,1,2024-09,5,140.0,0,0
1,1,2024-10,4,116.0,0,0
2,1,2024-11,7,208.0,0,0
3,1,2024-12,5,109.0,0,0
4,1,2025-01,1,41.0,0,0


## 3. Initial exploration

Review column names, data types, missing values, and basic numeric distributions before changing the data.

In [3]:
print("SUBSCRIPTIONS INFO")
subscriptions.info()

print("\nUSAGE INFO")
usage.info()

print("\nSUBSCRIPTION MISSING VALUES")
display(subscriptions.isna().sum().to_frame("missing_values"))

print("\nUSAGE MISSING VALUES")
display(usage.isna().sum().to_frame("missing_values"))

print("\nUSAGE NUMERIC SUMMARY")
display(usage[[
    "workouts_completed",
    "minutes_active",
    "classes_booked",
    "support_tickets"
]].describe())

SUBSCRIPTIONS INFO
<class 'pandas.DataFrame'>
RangeIndex: 4200 entries, 0 to 4199
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer_id          4200 non-null   int64  
 1   signup_date          4200 non-null   str    
 2   plan_type            4200 non-null   str    
 3   plan_tier            4200 non-null   str    
 4   monthly_price        4200 non-null   float64
 5   acquisition_channel  4200 non-null   str    
 6   primary_device       4200 non-null   str    
 7   age_group            4200 non-null   str    
 8   churned              4200 non-null   str    
 9   churn_date           1768 non-null   str    
 10  cancel_reason        1386 non-null   str    
dtypes: float64(1), int64(1), str(9)
memory usage: 572.0 KB

USAGE INFO
<class 'pandas.DataFrame'>
RangeIndex: 30685 entries, 0 to 30684
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------      

,missing_values
customer_id,0
signup_date,0
plan_type,0
plan_tier,0
monthly_price,0
acquisition_channel,0
primary_device,0
age_group,0
churned,0
churn_date,2432



USAGE MISSING VALUES


,missing_values
customer_id,0
month,0
workouts_completed,0
minutes_active,506
classes_booked,0
support_tickets,0



USAGE NUMERIC SUMMARY


,workouts_completed,minutes_active,classes_booked,support_tickets
count,30685.000000,30179.000000,30685.000000,30685.000000
mean,4.464722,134.047218,0.724393,0.128304
std,4.096791,131.166001,1.432328,0.370582
min,0.000000,0.000000,0.000000,0.000000
25%,1.000000,40.000000,0.000000,0.000000
50%,3.000000,100.000000,0.000000,0.000000
75%,6.000000,189.000000,1.000000,0.000000
max,48.000000,1653.000000,16.000000,4.000000


## 4. Data quality checks

### 4.1 Duplicate records and customer relationships

The subscription table should have one row per customer. In the usage table, the expected grain is one row per `customer_id` and `month`.

In [4]:
print("Duplicate subscription rows:", subscriptions.duplicated().sum())
print("Duplicate subscription customer IDs:", subscriptions["customer_id"].duplicated().sum())
print("Duplicate usage rows:", usage.duplicated().sum())
print(
    "Duplicate customer-month pairs:",
    usage.duplicated(subset=["customer_id", "month"]).sum()
)

subscription_ids = set(subscriptions["customer_id"])
usage_ids = set(usage["customer_id"])

print("\nCustomers without usage records:", len(subscription_ids - usage_ids))
print("Usage IDs missing from subscriptions:", len(usage_ids - subscription_ids))

Duplicate subscription rows: 0
Duplicate subscription customer IDs: 0
Duplicate usage rows: 160
Duplicate customer-month pairs: 160

Customers without usage records: 0
Usage IDs missing from subscriptions: 0


### 4.2 Inspect categorical values and missing activity minutes

In [5]:
categorical_columns = [
    "plan_type",
    "plan_tier",
    "acquisition_channel",
    "primary_device",
    "age_group",
    "churned"
]

for column in categorical_columns:
    print(f"\n{column.upper()}")
    display(subscriptions[column].value_counts(dropna=False).to_frame("count"))

print("\nRows with missing minutes_active")
display(usage.loc[usage["minutes_active"].isna()].head(10))


PLAN_TYPE


,count
plan_type,
monthly,2661
annual,1539



PLAN_TIER


,count
plan_tier,
basic,1929
plus,1462
premium,708
Basic,47
Plus,39
Premium,15



ACQUISITION_CHANNEL


,count
acquisition_channel,
organic,1298
paid_social,989
promo_offer,949
referral,488
partner,476


... (4 outputs left out)


CHURNED


,count
churned,
no,2432
yes,1768



Rows with missing minutes_active


,customer_id,month,workouts_completed,minutes_active,classes_booked,support_tickets
22,5,2025-11,0,NaN,0,0
33,7,2024-12,3,NaN,0,0
61,11,2025-10,9,NaN,0,0
81,14,2025-08,7,NaN,0,1
203,31,2025-03,0,NaN,0,1
386,60,2025-07,7,NaN,2,0
396,61,2025-08,1,NaN,0,1
418,66,2025-01,2,NaN,0,0
536,81,2025-09,0,NaN,0,0
605,92,2025-10,2,NaN,0,0


### 4.3 Investigate duplicate customer-month records

Before removing duplicate customer-month pairs, check whether the repeated records contain conflicting activity values. The observed duplicate pairs in this dataset had no conflicting usage values, so retaining the first row is appropriate.

In [6]:
duplicate_rows = (
    usage.loc[
        usage.duplicated(subset=["customer_id", "month"], keep=False)
    ]
    .sort_values(["customer_id", "month"])
)

usage_columns = [
    "workouts_completed",
    "minutes_active",
    "classes_booked",
    "support_tickets"
]

pair_variations = (
    duplicate_rows
    .groupby(["customer_id", "month"])[usage_columns]
    .nunique(dropna=False)
)

conflicting_pairs = pair_variations[pair_variations.gt(1).any(axis=1)]

print("Rows involved in duplicate customer-month pairs:", len(duplicate_rows))
print("Duplicate customer-month pairs:", duplicate_rows[["customer_id", "month"]].drop_duplicates().shape[0])
print("Pairs with conflicting usage values:", len(conflicting_pairs))

display(duplicate_rows.head(10))

Rows involved in duplicate customer-month pairs: 320
Duplicate customer-month pairs: 160
Pairs with conflicting usage values: 0


,customer_id,month,workouts_completed,minutes_active,classes_booked,support_tickets
286,44,2025-11,0,0.0,0,0
22367,44,2025-11,0,0.0,0,0
570,89,2024-12,5,206.0,3,0
7619,89,2024-12,5,206.0,3,0
757,116,2025-08,8,273.0,4,0
2624,116,2025-08,8,273.0,4,0
1107,167,2025-03,11,405.0,0,0
12289,167,2025-03,11,405.0,0,0
1221,182,2025-08,1,25.0,1,0
16963,182,2025-08,1,25.0,1,0


### 4.4 Investigate missing `minutes_active`

Do not automatically replace missing activity minutes with zero. A missing value is not proof that the customer had zero active minutes. Keep these values missing and document them.

In [7]:
missing_minutes = usage.loc[usage["minutes_active"].isna()]

print("Missing minutes_active in raw usage:", len(missing_minutes))

print("\nOther activity fields for rows with missing minutes_active")
display(
    missing_minutes[
        ["workouts_completed", "classes_booked", "support_tickets"]
    ].describe()
)

print("\nMissing minutes by workouts completed")
display(missing_minutes["workouts_completed"].value_counts().sort_index())

Missing minutes_active in raw usage: 506

Other activity fields for rows with missing minutes_active


,workouts_completed,classes_booked,support_tickets
count,506.000000,506.000000,506.000000
mean,4.328063,0.658103,0.146245
std,3.871076,1.267922,0.370116
min,0.000000,0.000000,0.000000
25%,1.250000,0.000000,0.000000
50%,3.000000,0.000000,0.000000
75%,6.000000,1.000000,0.000000
max,26.000000,7.000000,2.000000



Missing minutes by workouts completed


workouts_completed
0     55
1     72
2     65
3     68
4     48
5     37
6     49
7     32
8     23
9     13
10    11
11     9
12     7
13     1
14     4
15     3
17     4
21     2
22     1
23     1
26     1
Name: count, dtype: int64

## 5. Clean the datasets

Create working copies, standardize selected text fields, parse dates, and remove duplicate customer-month records. The duplicate check above should show zero conflicting pairs before deduplication.

In [8]:
subscriptions_clean = subscriptions.copy()
usage_clean = usage.copy()

# Standardize plan tier and churn labels.
subscriptions_clean["plan_tier"] = (
    subscriptions_clean["plan_tier"].str.strip().str.lower()
)
subscriptions_clean["churned"] = (
    subscriptions_clean["churned"].str.strip().str.lower()
)

# Parse date fields.
subscriptions_clean["signup_date"] = pd.to_datetime(
    subscriptions_clean["signup_date"], errors="coerce"
)
subscriptions_clean["churn_date"] = pd.to_datetime(
    subscriptions_clean["churn_date"], errors="coerce"
)
usage_clean["month"] = pd.to_datetime(
    usage_clean["month"], format="%Y-%m", errors="coerce"
)

# Replace missing cancellation reasons with an explicit label in a new column.
# The original cancel_reason column is retained.
subscriptions_clean["cancel_reason_clean"] = (
    subscriptions_clean["cancel_reason"].fillna("Not recorded")
)

# Remove repeated customer-month records only after checking for conflicts.
if len(conflicting_pairs) > 0:
    raise ValueError(
        "Conflicting duplicate customer-month pairs exist. "
        "Review them before removing duplicates."
    )

rows_before_deduplication = len(usage_clean)
removed_rows = usage_clean.loc[
    usage_clean.duplicated(subset=["customer_id", "month"], keep="first")
].copy()

usage_clean = usage_clean.drop_duplicates(
    subset=["customer_id", "month"],
    keep="first"
).copy()

print("Usage rows before deduplication:", rows_before_deduplication)
print("Duplicate rows removed:", len(removed_rows))
print("Usage rows after deduplication:", len(usage_clean))
print(
    "Missing minutes among removed duplicate rows:",
    removed_rows["minutes_active"].isna().sum()
)

Usage rows before deduplication: 30685
Duplicate rows removed: 160
Usage rows after deduplication: 30525
Missing minutes among removed duplicate rows: 3


## 6. Validate the cleaned datasets

These checks confirm that the transformations did not create invalid customer relationships, duplicate customer-month records, or impossible dates. Any remaining missing values are reported rather than silently imputed.

In [9]:
print("PLAN TIER COUNTS")
display(subscriptions_clean["plan_tier"].value_counts(dropna=False).to_frame("count"))

print("\nCHURN LABEL COUNTS")
display(subscriptions_clean["churned"].value_counts(dropna=False).to_frame("count"))

print("\nDATE VALIDATION")
print("Invalid signup dates:", subscriptions_clean["signup_date"].isna().sum())
print("Invalid usage months:", usage_clean["month"].isna().sum())
print(
    "Churned customers without churn dates:",
    subscriptions_clean.loc[
        subscriptions_clean["churned"].eq("yes"), "churn_date"
    ].isna().sum()
)

invalid_churn_dates = subscriptions_clean.loc[
    subscriptions_clean["churned"].eq("yes")
    & (subscriptions_clean["churn_date"] < subscriptions_clean["signup_date"])
]
print("Churn dates before signup dates:", len(invalid_churn_dates))

usage_with_signup = usage_clean.merge(
    subscriptions_clean[["customer_id", "signup_date"]],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

usage_before_signup = usage_with_signup.loc[
    usage_with_signup["month"].dt.to_period("M")
    < usage_with_signup["signup_date"].dt.to_period("M")
]
print("Usage records before signup month:", len(usage_before_signup))

PLAN TIER COUNTS


,count
plan_tier,
basic,1976
plus,1501
premium,723



CHURN LABEL COUNTS


,count
churned,
no,2432
yes,1768



DATE VALIDATION
Invalid signup dates: 0
Invalid usage months: 0
Churned customers without churn dates: 0
Churn dates before signup dates: 0
Usage records before signup month: 0


In [10]:
print("NEGATIVE ACTIVITY VALUES")
negative_activity = usage_clean[
    (usage_clean["workouts_completed"] < 0)
    | (usage_clean["minutes_active"] < 0)
    | (usage_clean["classes_booked"] < 0)
    | (usage_clean["support_tickets"] < 0)
]
print("Rows with negative activity:", len(negative_activity))

print("\nNON-INTEGER COUNT FIELDS")
for column in ["workouts_completed", "classes_booked", "support_tickets"]:
    print(f"{column}:", usage_clean[column].dropna().mod(1).ne(0).sum())

print("\nNON-POSITIVE SUBSCRIPTION PRICES")
print("Rows:", subscriptions_clean.loc[subscriptions_clean["monthly_price"] <= 0].shape[0])

print("\nTEXT WHITESPACE CHECK")
text_columns = [
    "plan_type", "plan_tier", "acquisition_channel",
    "primary_device", "age_group", "churned"
]
for column in text_columns:
    whitespace_count = (
        subscriptions_clean[column].dropna()
        .astype(str)
        .ne(subscriptions_clean[column].dropna().astype(str).str.strip())
        .sum()
    )
    print(f"{column}: {whitespace_count}")

NEGATIVE ACTIVITY VALUES
Rows with negative activity: 0

NON-INTEGER COUNT FIELDS
workouts_completed: 0
classes_booked: 0
support_tickets: 0

NON-POSITIVE SUBSCRIPTION PRICES
Rows: 0

TEXT WHITESPACE CHECK
plan_type: 0
plan_tier: 0
acquisition_channel: 0
primary_device: 0
age_group: 0
churned: 0


### 6.1 Validate row counts, keys, and missing values

In [11]:
print("SUBSCRIPTIONS")
print("Shape:", subscriptions_clean.shape)
print("Unique customers:", subscriptions_clean["customer_id"].nunique())
print("Duplicate customer IDs:", subscriptions_clean["customer_id"].duplicated().sum())

print("\nUSAGE")
print("Shape:", usage_clean.shape)
print(
    "Unique customer-month pairs:",
    usage_clean[["customer_id", "month"]].drop_duplicates().shape[0]
)
print(
    "Duplicate customer-month pairs:",
    usage_clean.duplicated(["customer_id", "month"]).sum()
)

unknown_customers = ~usage_clean["customer_id"].isin(
    subscriptions_clean["customer_id"]
)
print("Usage records without a matching subscription:", unknown_customers.sum())

print("\nMISSING VALUES — SUBSCRIPTIONS")
display(subscriptions_clean.isna().sum().to_frame("missing_values"))

print("\nMISSING VALUES — USAGE")
display(usage_clean.isna().sum().to_frame("missing_values"))

SUBSCRIPTIONS
Shape: (4200, 12)
Unique customers: 4200
Duplicate customer IDs: 0

USAGE
Shape: (30525, 6)
Unique customer-month pairs: 30525
Duplicate customer-month pairs: 0
Usage records without a matching subscription: 0

MISSING VALUES — SUBSCRIPTIONS


,missing_values
customer_id,0
signup_date,0
plan_type,0
plan_tier,0
monthly_price,0
acquisition_channel,0
primary_device,0
age_group,0
churned,0
churn_date,2432



MISSING VALUES — USAGE


,missing_values
customer_id,0
month,0
workouts_completed,0
minutes_active,503
classes_booked,0
support_tickets,0


### 6.2 Validation notes

- The raw usage table contained **160 duplicate customer-month pairs**. These pairs were checked for conflicting values before deduplication.
- Missing `minutes_active` values are retained as missing; they are not assumed to mean zero activity.
- The `cancel_reason_clean` column labels missing cancellation reasons as `Not recorded`, while preserving the original `cancel_reason` field.
- Expected final dimensions from the cleaning workflow are **4,200 subscription rows** and **30,525 usage rows**. If your source files differ, investigate before proceeding.

In [12]:
assert subscriptions_clean["customer_id"].nunique() == len(subscriptions_clean), (
    "Subscription customer IDs are not unique."
)
assert usage_clean.duplicated(["customer_id", "month"]).sum() == 0, (
    "Duplicate customer-month pairs remain."
)
assert unknown_customers.sum() == 0, (
    "Some usage records do not match a subscription customer."
)

print("Core key and relationship checks passed.")

Core key and relationship checks passed.


## 7. Export the cleaned datasets

The CSV files are saved in `data/processed`. These are the versions to use in the subsequent MySQL and Power BI stages.

In [13]:
subscriptions_output = PROCESSED_DIR / "subscriptions_clean.csv"
usage_output = PROCESSED_DIR / "usage_clean.csv"

subscriptions_clean.to_csv(subscriptions_output, index=False)
usage_clean.to_csv(usage_output, index=False)

print("Saved:", subscriptions_output)
print("Saved:", usage_output)

Saved: F:\Customer Churn Analysis\05_Data\Processed\subscriptions_clean.csv
Saved: F:\Customer Churn Analysis\05_Data\Processed\usage_clean.csv


### 7.1 Re-read the exports and verify them

In [14]:
subscriptions_check = pd.read_csv(subscriptions_output)
usage_check = pd.read_csv(usage_output)

print("Exported subscriptions:", subscriptions_check.shape)
print("Exported usage:", usage_check.shape)
print(
    "Duplicate customer-month pairs in exported usage:",
    usage_check.duplicated(["customer_id", "month"]).sum()
)
print(
    "Missing activity minutes in exported usage:",
    usage_check["minutes_active"].isna().sum()
)

Exported subscriptions: (4200, 12)
Exported usage: (30525, 6)
Duplicate customer-month pairs in exported usage: 0
Missing activity minutes in exported usage: 503


## 8. Optional: import the cleaned CSVs into MySQL

This section is optional. Run it only after installing the required packages and creating the target database in MySQL. It imports the two processed CSVs into tables named `subscriptions_python` and `usage_python`.

**Security:** enter your database password interactively. Do not save passwords in notebook cells or outputs. `if_exists="replace"` replaces existing tables with the same names.

In [ ]:
# If required, install these packages in a separate setup step:
# %pip install sqlalchemy pymysql

from getpass import getpass
from sqlalchemy import URL, create_engine, text

mysql_user = input("MySQL username: ")
mysql_password = getpass("MySQL password: ")
database_name = input("Existing MySQL database name: ")

connection_url = URL.create(
    drivername="mysql+pymysql",
    username=mysql_user,
    password=mysql_password,
    host="localhost",
    port=3306,
    database=database_name
)

engine = create_engine(connection_url)

with engine.connect() as connection:
    selected_database = connection.execute(
        text("SELECT DATABASE()")
    ).scalar()

print("Connected to database:", selected_database)

In [ ]:
# Read the exported clean datasets for import.
subscriptions_to_import = pd.read_csv(subscriptions_output)
usage_to_import = pd.read_csv(usage_output)

print("Subscriptions to import:", subscriptions_to_import.shape)
print("Usage to import:", usage_to_import.shape)

subscriptions_to_import.to_sql(
    name="subscriptions_python",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=500
)

usage_to_import.to_sql(
    name="usage_python",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=500
)

print("Both tables imported into MySQL.")